[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tensorchiefs/dl_course_2025/blob/main/notebooks/01_simple_forward_pass_keras_torch.ipynb)


## The basic mechanics of a neural network

In this notebook you will do a simple forward pass for a neural network with and without hidden layers by hand. This will familiarize you with the elemental flow of numeric values through the network.

In the second part `Keras` is introduced, which provides the "lego bricks" of deep learning. The same forward pass is repeated using the framework for `Keras`.


**Content:**

* calculate the forward pass of the neural network without hidden layer by hand, with matrix multiplication and keras
* visualize the learned decision boundary in a 2D plot
* calculate the forward pass of the neural network with one hidden layer (8 nodes) with matrix multiplication and keras
* visualize the learned decision boundary in a 2D plot
* compare the decision boundaries of the two models

#### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import ipywidgets as widgets
from IPython.display import display, clear_output

from typing import Callable

### Forward pass by hand

In [ ]:
# This defines a simple sigmoid function: That's the activation function of our
# neuron
def sigmoid(x):
    return (1 / (1 + np.exp(-x)))

In [ ]:
# Let's plot it
x = np.linspace(-10,10,100)
y = sigmoid(x)

plt.plot(x,y)
plt.grid()
plt.axvline(0, color="red")
plt.xlabel("z")
plt.ylabel("$\sigma(z)$")
plt.yticks(np.arange(0,1.1,.1));

Let's consider the most basic neural network: two inputs, one output. The inputs are numeric values, the output is supposed predict a two-level target, with values `0` or `1`. Here, the output will be interpreted as the probability that the target has value `1`.

This could also be framed as a logistic regression model with two numeric inputs.

In [ ]:
# Let's assume some input values x1 and x2
x1 = 1
x2 = 2.2

In [ ]:
# Let's also just assume some given weights and a bias
w1 = 0.3
w2 = 0.1
b  = 0

In [ ]:
(x1*w1+x2*w2)+b # output before the activation

In [ ]:
sigmoid((x1*w1+x2*w2)+b) ## output after the sigmoid activation

# We interpret the result as the probability of the small network
# to consider the input as belonging to a category `1`.

### Forward pass with matrix multiplication

In [ ]:
# Create a row vector of the input
X=np.array([x1,x2]).reshape(1,2)

In [ ]:
# Create a column vector of weights
W=np.array([w1, w2]).reshape(2,1)

In [ ]:
print(f"Input tensor:\n{X}, \nshape {X.shape}")
print()
print(f"Weight tensor:\n{W}, \nshape {W.shape}")


In [ ]:
# Output before the activation using a matrix multiplication
np.matmul(X,W)+b

In [ ]:
# Output after the sigmoid activation
# probability of the input being associated with class `1`
sigmoid(np.matmul(X,W)+b)

In [ ]:
# The function packs the full calculation into a one-liner
def predict_no_hidden(X):
    return sigmoid(np.matmul(X,W)+b)

In [ ]:
# This is an auxiliary function that is used to plot the decision boundary of a classifier
from matplotlib.colors import TwoSlopeNorm
def plotModel(predict: Callable, title:str)-> None:
    # define a grid for the 2D feature space
    # predict at each grid point the probability for class 1

    x1list = np.linspace(-10, 10, 10) # Define 100 points on the x1-axis
    x2list = np.linspace(-10, 10, 10) # Define 100 points on the x2-axis
    X1_grid, X2_grid = np.meshgrid(x1list, x2list)

    # model.predict for respective value x1 and x2
    p = np.array([predict(np.reshape(np.array([l1,l2]),(1,2))) for l1,l2 in zip(np.ravel(X1_grid), np.ravel(X2_grid))])
    if len(p.shape) == 3 and p.shape[2]==2:
        p = p[:,:,1] # pick p for class 1 if there are more than 2 classes
    p = np.reshape(p,X1_grid.shape)


    # visualize the predicted probabilities in the 2D feature space
    plt.figure(figsize=(16,4))
    plt.subplot(1,2,(1))

    cp = plt.contourf(X1_grid, X2_grid, p,cmap='RdBu_r')

    # uncomment if you want  a finer granularity of the decision  boundry
    levels = np.linspace(0, 1, num=31)
    norm = TwoSlopeNorm(vmin=p.min(), vcenter=0.5, vmax=p.max())
    cp = plt.contourf(X1_grid, X2_grid, p, levels=levels, cmap='RdBu_r', norm=norm)
    # Ensure that colorbar is centered at 0.5


    cbar = plt.colorbar(cp)
    cbar.set_ticks(np.arange(0,1.1,.1))
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')

#### 🔧 **YOUR TASK:**

Play around with the values for `x1` and `x2`

- What would you consider the decision boundary here? What output probabilities do you observe there?
- What is the shape of the decision boundary?


In [ ]:
# Do not care about this cell, just move the sliders and observe the predicted probability
def move_observation(x1, x2):
    clear_output(wait=True)
    plotModel(predict=predict_no_hidden, title='FCNN Separation without Hidden Layer')
    plt.scatter(x1, x2, c="black", s=50)
    print(f'predicted proba: {predict_no_hidden((x1,x2))}')
    plt.show()
    return None

x1_slider = widgets.FloatSlider(min=-10, max=10, step=.1, value=5, description='x1')
x2_slider = widgets.FloatSlider(min=-10, max=10, step=.1, value=3, description='x2')
widgets.interact(move_observation, x1=x1_slider, x2=x2_slider);

<details>
  <summary>🔑 Click here to View Answers:</summary>
  
- By playing around with x1 and x2, the predicted probality is in the range definded by the decision boundry

- The decision boundries are straight lines (planar decision boundries) in the variable space

</details>

### Forward pass with hidden layer (matrix multiplication)

We will now perform a forward pass through a randomly generated neural network with one hidden layer consisting of 8 nodes.


In [ ]:
# we use the same values for x1 and x2 and random normal values for the weights
X=np.array([[x1,x2]])

# We'll be investigating a random neural network. To make results reproducible we
# seed the RNG here
np.random.seed(23)

# Create the weights from the two input neurons to the 8 hidden neurons (2x8=16 weights)
#W1=np.reshape((np.random.normal(0,1,16)),(2,8))
W1 = np.random.normal(0,1,16).reshape(2,8)

# Create the biases for the 8 hidden neurons
#b1=np.reshape((np.random.normal(0,1,8)),(8,))
b1=np.random.normal(0,1,8).reshape(8,)

# Create the weights from the 8 hidden neurons to the single output neuron
#W2=np.reshape((np.random.normal(0,1,8)),(8,1))
W2=np.random.normal(0,1,8).reshape(8,1)

# Create the bias for the single output neuron
# b2=np.reshape((np.random.normal(0,1,1)),(1,))
b2=np.random.normal(0,1,1).reshape(1,)

In [ ]:
print(X.shape)
print(W1.shape)
print(b1.shape)
print(W2.shape)
print(b2.shape)

In [ ]:
hidden=sigmoid(np.matmul(X,W1)+b1)
hidden

In [ ]:
p_out=sigmoid(np.matmul(hidden,W2)+b2)
p_out

In [ ]:
def predict_hidden(X: np.ndarray) -> np.ndarray:
    """Compute the probability output of a hidden layer neural network

    Here's our own custom implementation of a neural network with two-inputs, a sigle hidden
    layer with 8 neurons and a single output.

    The "weights" of the network are stored in the matrices and vectors `W1`, `b1`, `W2`
    and `b2`.

    Parameters
    ----------
    X : np.ndarray
        n x 2 array of inputs tuples.

    Returns
    -------
    np.ndarray
        Shape n x 1 numpy array with computed outputs
    """
    hidden=sigmoid(np.matmul(X,W1)+b1)
    return(sigmoid(np.matmul(hidden,W2)+b2))

In [ ]:
# Here's our own custom implementation of a neural network with two-inputs, a sigle hidden
# layer with 8 neurons and a single output.
#
# Note:
# - `W1` (2x8) are the weights from the two inputs to the 8 hidden neurons.
# - `b1` (8x1) are the bias value for the 8 hidden neurons
# - `W2` (8x1) are the weights from the 8 hidden neurons to the single output neuron.
# - `b2` (1x1) is the weight for the final neuron
#
def predict_hidden(X):
    hidden=sigmoid(X @ W1 + b1)
    return(sigmoid(hidden @ W2 + b2))

In [ ]:
predict_hidden(np.array([[1,2]]))

In [ ]:
plotModel(predict_hidden, title='fcnn separation with hidden layer')
plt.scatter(x1,x2,c="black",s=50)

 - With only one hidden layer the decison boundries are not straight lines anymore!

#### 🔧 **YOUR TASK:**

- Add a second hidden Layer, with 8 nodes. You can choose random weights and biases (using `np.randon.normal()`).
- How does the Decision boundry look?

In [ ]:
# we use the same values for x1 and x2 and random normal values for the weights

# Data Array
X=np.array([[x1,x2]])

# Weigths
np.random.seed(23)
W1_=np.reshape((np.random.normal(0,1,16)),(2,8))
b1_=np.reshape((np.random.normal(0,1,8)),(8,))
W2_=np.reshape((np.random.normal(0,1,64)),(8,8))
b2_=np.reshape((np.random.normal(0,1,8)),(8,))
W3_=np.reshape((np.random.normal(0,1,8)),(8,1))
b3_=np.reshape((np.random.normal(0,1,1)),(1,))

# Print the shapes of the input, weights, and biases
print(f'X shape:   {X.shape}')
print(f'W1_ shape: {W1_.shape}')
print(f'b1_ shape: {b1_.shape}')
print(f'W2_ shape: {W2_.shape}')
print(f'b2_ shape: {b2_.shape}')
print(f'W3_ shape: {W3_.shape}')
print(f'b3_ shape: {b3_.shape}')

In [ ]:
# Hidden layer 1
hidden1=sigmoid(np.matmul(X,W1_)+b1_)
print(hidden1)

####  🔧 YOUR CODE HERE  #########
# Hidden layer 2



##########################

In [ ]:
# @title 🔑 Solution Code { display-mode: "form" }

# Hidden layer 2 , output of hidden 1 goes in here as input
hidden2=sigmoid(np.matmul(hidden1,W2_)+b2_)
print(hidden2)

# Applying the sigmoid activation function
p_out=sigmoid(np.matmul(hidden2,W3_)+b3_)
p_out
# Function to return the probability output after two hidden layers
def predict_hidden(X: np.ndarray):
    hidden1=sigmoid(np.matmul(X,W1_)+b1_)
    hidden2=sigmoid(np.matmul(hidden1,W2_)+b2_)
    return(sigmoid(np.matmul(hidden2,W3_)+b3_))
X,predict_hidden(X)
plotModel(predict_hidden, title='fcnn separation with two hidden layer')
plt.scatter(x1,x2,c="black",s=50)

<details>
  <summary>🔑 Click here to View Answers:</summary>


- The Decision boundry is now even more differnetly shaped as in the 1 hidden layer example. It divides the space into more diverse regions compared to when using no hidden layers.

</details>

## Visualizing the loss function

In the context of supervised Machine Learning, the focus is to find the neural network weights given some training data `X` and `y`.

We will now create an example dataset: It contains inputs tuples (`x1`,`x2`) and an associated target variable `y`.
The data conists of a `y=1` cloud centered around `x1,x2 = (2,1.5)` and a `y=0` cloud centered around `x1,x2 = (-2, -1.5)`.


In [ ]:
Xa = np.random.randn(100,2) * 2.5 + np.array([2,1.5]).reshape(1,2)
ya = np.full(100, fill_value=1).reshape(100,1)

Xb = np.random.randn(100,2) * 2.5 + np.array([-2,-1.5]).reshape(1,2)
yb = np.full(100, fill_value=0).reshape(100,1)

# Row-concatenate the two classes into a single dataset
X_demo = np.r_[Xa, Xb]
y_demo = np.r_[ya, yb]

print(X_demo[:3,:])
print(y_demo[:3])

In [ ]:
# Visualize the dataset
fig, ax = plt.subplots()

idx_f = [np.where(y_demo==1)]
idx_r = [np.where(y_demo==0)]

ax.scatter(X_demo[idx_r,0],X_demo[idx_r,1], alpha=0.7, label='0', s=10)
ax.scatter(X_demo[idx_f,0],X_demo[idx_f,1], alpha=0.7, label='1', s=10)
ax.set_title("Real and fake banknotes")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.legend(loc='upper left', fontsize=10)
ax.grid()

The **negative log loss** NLL (also known as binary cross-entropy) is defined as follows:


$$
\mathcal{L} = -\frac{1}{n} \sum_{i=1}^n \left[ y_i \log(p_i) + (1 - y_i)\log(1 - p_i) \right]
$$

If we have a vector of targets, `y`, and a vector of predicted probabilities for those targets, `p`, the following function will compute the NLL.


In [ ]:
# This function compute the negative log-likelihood (NLL) given some target `y` and
# corresponding predicted probabilities `p`.
def compute_NLL(p, y):
    # Clip probabilities to avoid log(0) when computing log(p) or log(1-p)
    p = np.clip(p, 1e-15, 1-1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p), axis=0)

In [ ]:
def compute_NLL(p: np.ndarray, y: np.ndarray) -> np.floating:
    """Compute the Negative Log Likelihood

    Compute the Negative Log Likelihood for a given target and its corresponding predicted
    probability

    Parameters
    ----------
    p : np.ndarray
        n x K array of predicted probabilities
        There will be n predicted probabilities. If K>1, then the can contain the output
        of K models.
    y : np.ndarray
        n x 1 array of target variables (0 or 1).

    Returns
    -------
    float
        The computed Negative Log Likelihood.

    Notes
    -----

    The broadcasting behaviour of numpy ensures that the function will run correctly, no matter
    whether `p` is a vector or a matrix (i.e. `K` separate predictions for all `n` points). This
    is an example of vectorized code.
    """

    # Clip probabilities to avoid log(0) when computing log(p) or log(1-p)
    p = np.clip(p, 1e-15, 1-1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p), axis=0)


In [ ]:

# ℹ️ You don't have to understand this cell in detail.
# The aim of this cell is to compute the NLL (loss) given the data FOR A BUNCH OF DIFFERENT
# NEURAL NETWORK PARAMETRIZATIONS!

# Predict the probability given a list of 2D inputs (in nx2 matrix `X`) and a list of
# possible w1,w2 parametrizations (given in mx2 matrix `W`).
# It relies on numpy's matrix multiplication broadcasting behaviour.
def predict_no_hidden_generic(X, W, b=0) -> np.ndarray:
    return sigmoid(np.matmul(X,W)+b)


# Define a grid of `w1` and `w2` values
w1list = np.linspace(-4, 4, 50) # Define 100 points on the w1-axis
w2list = np.linspace(-4, 4, 50) # Define 100 points on the w2-axis
W1_grid, W2_grid = np.meshgrid(w1list, w2list)

# `W_grid` has shape (2,100). Each of the 100 columns contains two weights that
# represent one simple, small network. We are thus creating 100 different models.
W_grid = np.r_[
    W1_grid.reshape(1, 50*50),
    W2_grid.reshape(1, 50*50)
]

# Predict the probabilities for each of the 100 models (described in `W_grid`)
# The i-th model is encoded in `W_grid[:,i]`.
# `p` contains the predicted probabilities of model `i` in `p[:,i]`
p = predict_no_hidden_generic(X_demo, W_grid, b=0)

# Now compute the NLL loss for each of the models
nll_grid = compute_NLL(p, y_demo)



In [ ]:
# 🧪 You can ignore this cell, just run it an look at its output


# Reshape to match the grid shape (necessary for `plt.contourf`)
nll_grid = nll_grid.reshape(W1_grid.shape)
# Find minimum in nll_grid
i_r,i_c = np.unravel_index(np.argmin(nll_grid), nll_grid.shape)
w1_sol = w1list[i_c]
w2_sol = w2list[i_r]
nll_sol = np.min(nll_grid)

# Visualize the predicted probabilities in the 2D feature space
# plt.figure(figsize=(16,4))
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

levels = np.linspace(0, 20, num=101)
cp = ax.contourf(W1_grid, W2_grid, nll_grid, levels=levels, cmap='viridis_r')
# cp = ax.plot_wireframe(W1_grid, W2_grid, nll_grid, )
h, = ax.plot(w1_sol, w2_sol, nll_sol, "rx", zorder=99)
ax.legend([h], ["Best solution"])
cbar = fig.colorbar(cp)
ax.set_xlabel('w1')
ax.set_ylabel('w2')
ax.view_init(elev=25, azim=10, roll=0)
ax.grid()

The visualization above shows us the **loss surface** over the space of all neural network weights. Since we only have two weights here, we can actually look at it and understand it.

The red cross shows the w1,w2 combination with the lowest loss.

*Note:* The fact that the loss surface is 2D **and** simple and convex is because we're dealing with the most simple neural network architecture.

Below, we show the decision boundary of the neural network with the optimal weights w1,w2 as seen above:

In [ ]:
# Plot the decision boundary
W = np.array([w1_sol, w2_sol]).reshape(2,1)
def predict_custom(X):
    return predict_no_hidden_generic(X, W)


plotModel(predict_custom, title='Output of optimal model')
plt.scatter(X_demo[idx_r,0],X_demo[idx_r,1], alpha=0.7, label='0', s=6)
plt.scatter(X_demo[idx_f,0],X_demo[idx_f,1], alpha=0.7, label='1', s=6)
plt.grid()


TODO: Illustrate how the this simple model is fit (?)
https://playground.tensorflow.org

# Keras
<img src="https://keras.io/img/logo.png
" alt="Sample Image" width="150">


[Visit Keras Documentation](https://keras.io/api/)

**Look at this part after the introduction of Keras.**


We now do the same as above using Keras.

In [ ]:
# imports
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import torch # not needed yet

print(f'Keras_version: {keras.__version__}')# 3.5.0
print(f'torch_version: {torch.__version__}')# 2.5.1+cu121
print(f'keras backend: {keras.backend.backend()}')

from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical
from keras import optimizers

### Forward pass in keras

In [ ]:
# Set up an empty neural network. Our network starts with two inputs
inputs = keras.Input(shape=(2,))
# Those inputs are then fed into a single-neuron layer, with activation "sigmoid"
# This is already our final layer and its output is thus the network output
output = Dense(1, activation="sigmoid")(inputs)

# Create the model
model = keras.Model(inputs, output)

model.summary()

In [ ]:
# Set the weights of the model. Note that before that, only the architecture
# of the model was known, not the actual values that sit on all edges.
model.set_weights([W,np.array([b])])

In [ ]:
# Plotting the decision boundary
plotModel(model.predict, 'fcnn separation without hidden layer with keras')
plt.scatter(X[0][0],X[0][1],c="black",s=50)

### Forward pass with hidden layer (keras)

In [ ]:
# We define a network with two inputs, one hidden layer with 8 neurons
# and a single output indicating probability that the class is `1`
input = keras.Input(shape=(2,))
x = Dense(8, activation="sigmoid")(input)
output = Dense(1, activation="sigmoid")(x)

model = keras.Model(input, output)

model.summary()

In [ ]:
model.set_weights([W1,b1,W2,b2]) ## set the weights of the model to W1, b1, W2 and b2

In [ ]:
# Plotting the decision boundary

plotModel(model.predict, 'fcnn separation with hidden layer keras')
plt.scatter(X[0][0],X[0][1],c="black",s=50)
# moving the x1 or x2 values along the lines to change the probas
plt.vlines(X[0][0],-10,10)
plt.hlines(X[0][1],-10,10)